# From episodes to tasks

## Point

You have started a new job and need to find a reliable commute. There are three routes. On the first morning, you try route A and arrive late. The next morning, route B also takes too long. Route C gets you there on time, so you use it again the following morning.

Each journey is an episode. The four journeys together are a task: experience from the early mornings helps you make better choices later. Someone who forgets each journey and keeps choosing route A stays late.

## Objective

Compare remembering with forgetting over four mornings. For this simple example, travel conditions stay fixed and only one route gets you to work on time.

| Level | In this example |
| --- | --- |
| Step | Choose a route and observe whether you arrive on time. |
| Episode | One morning's commute. |
| Task | Four successive commutes to the same workplace. |

The model treats a whole commute as one step; a navigation environment could use many steps per journey. Routes A, B, and C are actions 0, 1, and 2. Reward 1 means arriving on time; reward 0 means arriving late.


## Keep the routes fixed between commutes

A public `env.reset()` or `env.reset(seed=None)` starts a new task and draws the next episode seed from the wrapper RNG. Passing `options={"regenerate_routes": True}` to the public reset asks `CommuteRoutes` to choose which route gets you there on time for the new task.

Between episodes, the wrapper passes the next explicit episode seed without requesting new route conditions. This starts another morning while preserving the task. The option controls task generation separately from seeding. The observation is always 0 and `info` is empty: the agent must learn from its journeys rather than being told the best route.

With fixed route conditions, the base problem is a one-state MDP. The unknown route conditions make the learning problem partially observable. Remembering an earlier journey helps with a later one. See the [README's explanation](../README.md#from-an-mdp-to-a-partially-observable-learning-task).


In [ ]:
import gymnasium as gym

from repeat_task_gym import RepeatTaskEnv


class CommuteRoutes(gym.Env):
    """One commute per episode; route conditions persist within a task."""

    def __init__(self):
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Discrete(1)
        self._on_time_route = None

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if self._on_time_route is None or (options or {}).get("regenerate_routes", False):
            self._on_time_route = int(self.np_random.integers(3))
        return 0, {}

    def step(self, action):
        if not self.action_space.contains(action):
            raise ValueError(f"Invalid route: {action}")
        reward = float(action == self._on_time_route)
        return 0, reward, True, False, {}


## Remember between episodes

The agent starts with route A, tries another route after arriving late, and reuses a route after arriving on time. Its memory is `next_route`. The remembering version keeps it between mornings; the forgetting version starts again with route A.

After each episode, a separate `step(0)` consumes the reset frame for the following morning. Its action is ignored and its zero reward does not represent another commute.

`run_task` defaults to `seed=None`. We use `seed=7` for both runs below to compare the agents under the same route conditions. Each run starts with fresh memory and ends after four commutes.


In [ ]:
def run_task(*, remember, seed=None):
    env = RepeatTaskEnv(
        CommuteRoutes(), max_task_episodes=4,
        episode_reset_options={"regenerate_routes": False},
    )
    next_route = 0
    episode_returns = []
    try:
        env.reset(seed=seed, options={"regenerate_routes": True})
        while True:
            observation, reward, terminated, truncated, _ = env.step(next_route)
            assert observation["episode_terminated"] and not observation["episode_truncated"]
            episode_returns.append(reward)
            if terminated or truncated:
                break  # This succession of episodes, the task, is finished.

            # Use this episode's outcome to choose the next episode's action.
            if reward == 0:
                next_route = (next_route + 1) % 3
            if not remember:
                next_route = 0  # Forget at the episode boundary.

            observation, reward, terminated, truncated, _ = env.step(0)
            assert not observation["episode_terminated"] and not observation["episode_truncated"]
            assert reward == 0.0
            assert not terminated and not truncated
    finally:
        env.close()
    return episode_returns


# Replay the same task to compare two ways of handling episode boundaries.
remembering = run_task(remember=True, seed=7)
forgetting = run_task(remember=False, seed=7)
assert remembering == [0.0, 0.0, 1.0, 1.0]
assert forgetting == [0.0, 0.0, 0.0, 0.0]

print("Episode | Remembering | Forgetting")
for episode, (kept, cleared) in enumerate(zip(remembering, forgetting, strict=True), 1):
    print(f"{episode:7} | {kept:11.0f} | {cleared:10.0f}")
print(f"Task return, remembering: {sum(remembering):.0f}")
print(f"Task return, forgetting:  {sum(forgetting):.0f}")


## Read the journeys as one task

| Episode | Journey | What the agent learns |
| --- | --- | --- |
| 1 | Route A arrives late. | Try another route tomorrow. |
| 2 | Route B also arrives late. | Route C remains untried. |
| 3 | Route C arrives on time. | This route works. |
| 4 | Route C arrives on time again. | Use the earlier discovery. |

The remembering agent arrives on time twice; the forgetting agent is late every day. The early journeys helped the remembering agent improve its later journeys. This is the learning progress we want to measure within a task.

Next: [01 — Random rollout](01_random_rollout.ipynb) traces steps and episode boundaries. [02](02_task_termination.ipynb) chooses when the task ends, [03](03_rng_seeding_control.ipynb) explains resets, and [04](04_gymnasium_composition.ipynb) collects statistics.
